# First LLM Classifier

A complete notebook covering the full tutorial: prompting with Python, structured responses, reliability tips, bulk classification, evaluation, and prompt improvement.

Work through the cells in order. You will need a [Hugging Face](https://huggingface.co/) account and API token to run the code.

## Install packages

Run this once to install all required libraries into your environment.

The cell below uses [`uv`](https://docs.astral.sh/uv/), the package manager recommended for this tutorial. If you are using a standard pip environment, replace `!uv add` with `!pip install`.

In [ ]:
# If you are using uv (recommended):
!uv add huggingface_hub rich ipywidgets pydantic pandas tenacity scikit-learn matplotlib

# If you are using pip instead:
# !pip install huggingface_hub rich ipywidgets pydantic pandas tenacity scikit-learn matplotlib

## Imports

All libraries used throughout this notebook are imported here.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import batched
from typing import Literal

import pandas as pd
from huggingface_hub import InferenceClient
from pydantic import BaseModel
from rich import print
from rich.progress import track
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from tenacity import retry

## Authentication

Paste your Hugging Face API token below. You can find or create one at https://huggingface.co/settings/tokens.

> **Note:** In a production setting you would store the token in an environment variable rather than hard-coding it in a notebook.

In [ ]:
token = "Paste your key here"

In [ ]:
client = InferenceClient(token=token)

---

## Prompting with Python

Send your first prompt to a large-language model via Hugging Face's API.

### Basic prompt

In [ ]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of data journalism in a concise sentence",
        }
    ],
    model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
)

In [ ]:
print(response.choices[0].message.content)

### Try a different model — Gemma 3 from Google

In [ ]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of data journalism in a concise sentence",
        }
    ],
    model="google/gemma-3-27b-it",
)
print(response.choices[0].message.content)

### System prompts

A *system* prompt establishes the model's tone and role before any user message.

In [ ]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "you are an enthusiastic nerd who believes data journalism is the future.",
        },
        {
            "role": "user",
            "content": "Explain the importance of data journalism in a concise sentence",
        },
    ],
    model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
)
print(response.choices[0].message.content)

In [ ]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "you are a crusty, ill-tempered editor who hates math and thinks data journalism is a waste of time and resources.",
        },
        {
            "role": "user",
            "content": "Explain the importance of data journalism in a concise sentence",
        },
    ],
    model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
)
print(response.choices[0].message.content)

---

## Structured responses

LLMs can answer simple, factual questions — not just write essays. Here we classify professional sports teams by their league.

### Simple classification function

In [ ]:
def classify_team(name):
    prompt = """
You are an AI model trained to classify text.

I will provide the name of a professional sports team.

You will reply with the sports league in which they compete.
"""

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": name,
            },
        ],
        model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
    )

    return response.choices[0].message.content

In [ ]:
team_list = ["Chicago Cubs", "Chicago Bears", "Chicago Bulls"]

for team in team_list:
    league = classify_team(team)
    print([team, league])

### Validating responses with Pydantic

Use a Pydantic model and JSON schema to restrict the LLM to a fixed set of answers.

In [ ]:
class SportsLeague(BaseModel):
    answer: Literal["MLB", "NFL", "NBA", "Other"]

In [ ]:
def classify_team(name):
    prompt = """
You are an AI model trained to classify text.

I will provide the name of a professional sports team.

You will reply with the sports league in which they compete.

If the team's league is not in the provided options, reply with "Other".
"""

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": name,
            },
        ],
        model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "SportsLeague",
                "schema": SportsLeague.model_json_schema(),
            },
        },
    )

    result = SportsLeague.model_validate_json(response.choices[0].message.content)
    return result.answer

In [ ]:
for team in team_list:
    league = classify_team(team)
    print([team, league])

In [ ]:
classify_team("Chicago Blackhawks")  # NHL — not in our list, should return "Other"

---

## Improving reliability

Three techniques that make classifiers more consistent: lowering temperature, few-shot prompting, and automatic retries.

### Temperature = 0  (less creativity, more consistency)

In [ ]:
def classify_team(name):
    prompt = """
    You are an AI model trained to classify text.

    I will provide the name of a professional sports team.

    You will reply with the sports league in which they compete.

    If the team doesn't belong in the provided sports league options, reply with "Other".
    """

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": name,
            },
        ],
        model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "SportsLeague",
                "schema": SportsLeague.model_json_schema(),
            },
        },
        temperature=0,
    )

    result = SportsLeague.model_validate_json(response.choices[0].message.content)
    return result.answer

### Few-shot prompting

Prime the LLM with example input/output pairs before your real request.

In [ ]:
def classify_team(name):
    prompt = """
    You are an AI model trained to classify text.

    I will provide the name of a professional sports team.

    You will reply with the sports league in which they compete.

    If the team doesn't belong in the provided sports league options, reply with "Other".
    """

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user",      "content": "Los Angeles Rams"},
            {"role": "assistant", "content": '{"answer": "NFL"}'},
            {"role": "user",      "content": "Los Angeles Dodgers"},
            {"role": "assistant", "content": '{"answer": "MLB"}'},
            {"role": "user",      "content": "Los Angeles Lakers"},
            {"role": "assistant", "content": '{"answer": "NBA"}'},
            {"role": "user",      "content": "Los Angeles Kings"},
            {"role": "assistant", "content": '{"answer": "Other"}'},
            {"role": "user",      "content": name},
        ],
        model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "SportsLeague",
                "schema": SportsLeague.model_json_schema(),
            },
        },
        temperature=0,
    )

    result = SportsLeague.model_validate_json(response.choices[0].message.content)
    return result.answer

### Automatic retries with tenacity

The `@retry` decorator automatically re-runs the function if an exception occurs, making your classifier resilient to temporary network or API failures.

In [ ]:
@retry
def classify_team(name):
    prompt = """
    You are an AI model trained to classify text.

    I will provide the name of a professional sports team.

    You will reply with the sports league in which they compete.

    If the team doesn't belong in the provided sports league options, reply with "Other".
    """

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user",      "content": "Los Angeles Rams"},
            {"role": "assistant", "content": '{"answer": "NFL"}'},
            {"role": "user",      "content": "Los Angeles Dodgers"},
            {"role": "assistant", "content": '{"answer": "MLB"}'},
            {"role": "user",      "content": "Los Angeles Lakers"},
            {"role": "assistant", "content": '{"answer": "NBA"}'},
            {"role": "user",      "content": "Los Angeles Kings"},
            {"role": "assistant", "content": '{"answer": "Other"}'},
            {"role": "user",      "content": name},
        ],
        model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "SportsLeague",
                "schema": SportsLeague.model_json_schema(),
            },
        },
        temperature=0,
    )

    result = SportsLeague.model_validate_json(response.choices[0].message.content)
    return result.answer

---

## Bulk prompts

Classify many items in a single API call by submitting batches, then process the entire dataset using sequential or parallel batching.

### Batch Pydantic model and sports team classifier

In [ ]:
class SportsLeagueList(BaseModel):
    answers: list[Literal["MLB", "NFL", "NBA", "Other"]]

In [ ]:
def classify_teams(name_list):
    prompt = """
You are an AI model trained to classify text.

I will provide a list of professional sports team names separated by newlines.

You will reply with the sports league in which they compete.

If the team's league is not on the list, you should label them as "Other".
"""

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": "Los Angeles Rams\nLos Angeles Dodgers\nLos Angeles Lakers\nLos Angeles Kings",
            },
            {
                "role": "assistant",
                "content": '{"answers": ["NFL", "MLB", "NBA", "Other"]}',
            },
            {"role": "user", "content": "\n".join(name_list)},
        ],
        model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "SportsLeagueList",
                "schema": SportsLeagueList.model_json_schema(),
            },
        },
        temperature=0,
    )

    result = SportsLeagueList.model_validate_json(response.choices[0].message.content)
    return dict(zip(name_list, result.answers))

In [ ]:
classify_teams(team_list)

### Campaign finance data

Now let's apply these techniques to a real dataset — California campaign finance expenditures.

In [ ]:
df = pd.read_csv(
    "https://palewi.re/docs/first-llm-classifier/_static/Form460ScheduleESubItem.csv"
)
df.sample(10)

### Payee Pydantic model

In [ ]:
class PayeeList(BaseModel):
    answers: list[Literal["Restaurant", "Bar", "Hotel", "Other"]]

### `classify_payees` — classify a batch of payees

In [ ]:
@retry
def classify_payees(name_list, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"):
    prompt = """
You are an AI model trained to categorize businesses based on their names.

You will be given a list of business names, each separated by a new line.

Your task is to analyze each name and classify it into one of the following categories: Restaurant, Bar, Hotel, or Other.

If a business does not clearly fall into Restaurant, Bar, or Hotel categories, you should classify it as "Other".

Even if the type of business is not immediately clear from the name, it is essential that you provide your best guess based on the information available to you. If you can't make a good guess, classify it as Other.
"""

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": "Intercontinental Hotel\nPizza Hut\nCheers\nWelsh's Family Restaurant\nKTLA\nDirect Mailing",
            },
            {
                "role": "assistant",
                "content": '{"answers": ["Hotel", "Restaurant", "Bar", "Restaurant", "Other", "Other"]}',
            },
            {
                "role": "user",
                "content": "Subway Sandwiches\nRuth Chris Steakhouse\nPolitical Consulting Co\nThe Lamb's Club",
            },
            {
                "role": "assistant",
                "content": '{"answers": ["Restaurant", "Restaurant", "Other", "Bar"]}',
            },
            {"role": "user", "content": "\n".join(name_list)},
        ],
        model=model,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "PayeeList",
                "schema": PayeeList.model_json_schema(),
            },
        },
        temperature=0,
    )

    result = PayeeList.model_validate_json(response.choices[0].message.content)
    assert len(result.answers) == len(name_list), \
        f"Expected {len(name_list)} answers but got {len(result.answers)}"
    return pd.DataFrame({"payee": name_list, "category": result.answers})

In [ ]:
sample_list = df.sample(10).payee
classify_payees(sample_list)

### `classify_batches` — sequential batching

In [ ]:
def classify_batches(name_list, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", batch_size=10, wait=1):
    """Split the provided list of names into batches and classify with our LLM one by one."""
    all_results = []
    batch_list = list(batched(list(name_list), batch_size))
    for batch in track(batch_list, description="Classifying batches..."):
        batch_df = classify_payees(list(batch), model)
        all_results.append(batch_df)
        time.sleep(wait)
    return pd.concat(all_results, ignore_index=True)

In [ ]:
bigger_sample = df.sample(100).payee
results_df = classify_batches(bigger_sample)

In [ ]:
results_df.head(10)

In [ ]:
results_df.category.value_counts()

### `classify_batches_parallel` — parallel batching

In [ ]:
def classify_batches_parallel(name_list, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", batch_size=10, max_workers=4):
    """Split the provided list of names into batches and classify with our LLM in parallel."""
    all_results = []
    batch_list = list(batched(list(name_list), batch_size))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(classify_payees, list(batch), model)
            for batch in batch_list
        ]
        for future in track(
            as_completed(futures),
            total=len(futures),
            description="Classifying batches...",
        ):
            batch_df = future.result()
            all_results.append(batch_df)
    return pd.concat(all_results, ignore_index=True)

In [ ]:
results_df = classify_batches_parallel(bigger_sample)

---

## Evaluation

Use a hand-labeled supervised sample and scikit-learn metrics to measure how well the LLM classifier performs, and compare it against a traditional machine-learning baseline.

### Load the supervised sample

In [ ]:
sample_df = pd.read_csv(
    "https://palewi.re/docs/first-llm-classifier/_static/sample.csv"
)
sample_df.head()

### Train / test split

We use 67 % of the sample for testing (the LLM needs no training data) and hold back 33 % as a training slice for few-shot prompting later.

In [ ]:
training_input, test_input, training_output, test_output = train_test_split(
    sample_df[["payee"]],
    sample_df["category"],
    test_size=0.67,
    random_state=42,
)

### Run the LLM on the test set

In [ ]:
llm_df = classify_batches(test_input.payee)
llm_df.head()

### Classification report

In [ ]:
print(classification_report(test_output, llm_df.category))

### Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(test_output, llm_df.category)

### Traditional machine-learning baseline

For comparison, train a `LinearSVC` model on the training slice and evaluate it on the same test set.

In [ ]:
vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    min_df=5,
    norm="l2",
    encoding="latin-1",
    ngram_range=(1, 3),
)
preprocessor = ColumnTransformer(
    transformers=[("payee", vectorizer, "payee")], sparse_threshold=0, remainder="drop"
)
pipeline = Pipeline(
    [("preprocessor", preprocessor), ("classifier", LinearSVC(dual="auto"))]
)

In [ ]:
model = pipeline.fit(training_input, training_output)

In [ ]:
predictions = model.predict(test_input)

In [ ]:
print(classification_report(test_output, predictions))

In [ ]:
ConfusionMatrixDisplay.from_predictions(test_output, predictions)

### Compare multiple LLM models

Loop through several models to see which performs best on this task.

In [ ]:
model_list = [
    "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
    "google/gemma-3-27b-it",
    "Qwen/Qwen3-235B-A22B-Instruct-2507",
]

for m in model_list:
    print(f"Model: {m}")
    result_df = classify_batches(test_input.payee, m)
    print(classification_report(test_output, result_df.category))

---

## Improvement

Learn from the model's mistakes and use the training data as few-shot prompts to boost accuracy.

### Find misclassifications

In [ ]:
comparison_df = llm_df.merge(
    sample_df, on="payee", how="inner", suffixes=["_llm", "_human"]
)
mistakes_df = comparison_df[
    comparison_df.category_llm != comparison_df.category_human
]
mistakes_df.head(10)

In [ ]:
mistakes_df.to_csv("mistakes.csv", index=False)

### Use training data as few-shot prompts

Convert the hand-labeled training slice into few-shot examples that prime the LLM before each request.

In [ ]:
def get_fewshots(training_input, training_output, batch_size=10):
    """Convert the training data into few-shot prompt messages."""
    input_batches = list(batched(training_input.payee, batch_size))
    output_batches = list(batched(training_output, batch_size))
    fewshot_list = []
    for input_list, output_list in zip(input_batches, output_batches):
        prompt = "\n".join(input_list)
        response = PayeeList(answers=list(output_list)).model_dump_json()
        fewshot_list.extend([
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ])
    return fewshot_list

In [ ]:
fewshot_list = get_fewshots(training_input, training_output)
fewshot_list[:2]

### Updated `classify_payees` using training data as few-shot examples

Swap the hardcoded examples for the generated few-shot list.

In [ ]:
@retry
def classify_payees(name_list, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"):
    prompt = """
You are an AI model trained to categorize businesses based on their names.

You will be given a list of business names, each separated by a new line.

Your task is to analyze each name and classify it into one of the following categories: Restaurant, Bar, Hotel, or Other.

If a business does not clearly fall into Restaurant, Bar, or Hotel categories, you should classify it as "Other".

Even if the type of business is not immediately clear from the name, it is essential that you provide your best guess based on the information available to you. If you can't make a good guess, classify it as Other.
"""

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": prompt},
            *get_fewshots(training_input, training_output),
            {"role": "user", "content": "\n".join(name_list)},
        ],
        model=model,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "PayeeList",
                "schema": PayeeList.model_json_schema(),
            },
        },
        temperature=0,
    )

    result = PayeeList.model_validate_json(response.choices[0].message.content)
    assert len(result.answers) == len(name_list), \
        f"Expected {len(name_list)} answers but got {len(result.answers)}"
    return pd.DataFrame({"payee": name_list, "category": result.answers})

### Re-evaluate with the improved prompt

In [ ]:
llm_df = classify_batches(
    test_input.payee,
    model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
)
print(classification_report(test_output, llm_df.category))